# GlutenGuard — Week 1: Core Engine (Text Input)

Month 3, Week 1 goal: build the core classification engine.

`classify_ingredient_list(text)` takes a raw ingredient list (plus any allergen/warning
statements) as a string, sends it to Claude with a celiac-safety system prompt, and returns
a structured `GlutenGuardResult` — Green / Yellow / Red, with an ingredient-by-ingredient
explanation, cross-contamination notes, and the mandatory verify-before-eating reminder.

Image input (the actual label-photo scanner) is next — not built yet. This notebook is
text-in, structured-JSON-out only, so the classification logic can be tested and tuned before
vision is wired up.

In [1]:
# Run once if the packages aren't installed yet
%pip install -q anthropic python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports
from textwrap import dedent
from typing import Literal, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from anthropic import Anthropic

In [10]:
# Client initialization

# Reads ANTHROPIC_API_KEY from a .env file (or the environment) in the working directory
load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

## Structured output schema

Passed to Claude as `output_format` via `client.messages.parse(...)` — the API constrains
generation to match this schema, so the response is always valid, parseable JSON rather than
something we have to regex out of free text.

In [4]:
class IngredientFinding(BaseModel):
    """Gluten-status verdict for a single ingredient or listed term."""

    ingredient: str = Field(description="The ingredient or term as it appears on the label")
    gluten_status: Literal["contains_gluten", "gluten_free", "uncertain"]
    reasoning: str = Field(description="One sentence explaining the verdict for this ingredient")


class GlutenGuardResult(BaseModel):
    """Structured GlutenGuard classification for one product label."""

    classification: Literal["Green", "Yellow", "Red"]
    summary: str = Field(description="One to two sentence plain-language explanation")
    ingredient_analysis: list[IngredientFinding]
    cross_contamination_warning: Optional[str] = Field(
        default=None,
        description="Any 'may contain', shared-facility, or shared-equipment language found "
        "on the label, or null if none is present",
    )
    reasons_for_classification: list[str] = Field(
        description="Bulleted list of the specific reasons driving the classification"
    )
    mandatory_reminder: str = Field(
        description="Reminder to verify the package before consuming"
    )

## System prompt
Uncertainty always defaults to Yellow, and Green is
never used unless every ingredient is unambiguously gluten-free with no contamination warning.

In [5]:
SYSTEM_PROMPT = dedent("""
    You are GlutenGuard, an assistant that helps people with celiac disease decide whether a
    packaged food is safe to eat, based on the ingredient list and any allergen or warning
    statements from the product label.

    You will be given the raw text of a product's ingredient list, and possibly accompanying
    allergen or warning statements (e.g. "Contains: Wheat", "May contain traces of gluten",
    "Manufactured in a facility that also processes wheat").

    Classify the product into exactly one of three categories:

    - Green: No gluten-containing ingredients and no gluten/wheat warnings of any kind. You are
      confident, based on the label alone, that the product is gluten-free.
    - Yellow: The label is ambiguous, incomplete, or contains an ingredient of uncertain gluten
      status (e.g. "natural flavors", "modified food starch" without a named source, "malt"
      without specifying the source, "dextrin"), OR the label includes a "may contain",
      shared-facility, or shared-equipment warning.
    - Red: The ingredient list contains an explicit gluten-containing ingredient (wheat,
      barley, rye, malt from barley, brewer's yeast, etc.) or an explicit "Contains: Wheat" /
      gluten allergen statement.

    CRITICAL SAFETY RULE — be conservative:
    - If you are not certain an ingredient is gluten-free, classify as Yellow. Never output
      Green unless every single ingredient is unambiguously gluten-free AND there is no
      cross-contamination warning.
    - Treat any ingredient name you do not recognize as uncertain, not as safe.
    - When genuinely uncertain, prefer Yellow over Red as well — only classify Red when there
      is direct, explicit evidence of gluten content in the ingredients or warnings. Do not
      guess your way to Red on a hunch.

    For every ingredient (or ambiguous term) in the list, provide a gluten-status verdict and a
    short reason. Quote or summarize any cross-contamination warning found on the label. List
    the specific reasons driving the overall classification. Always include a reminder that the
    user must still verify the package, manufacturer information, and certified gluten-free
    status before consuming — this assessment is not a substitute for that.
""").strip()

## Core function

`mandatory_reminder` is overwritten with a fixed, hardcoded string after the model responds —
this is a safety-critical disclaimer, so it must not depend on the model remembering to include
it (or phrasing it responsibly) every time.

In [6]:
MANDATORY_REMINDER = (
    """
    This is not a substitute for checking the package yourself. Before eating, verify the 
    ingredient list and allergen statement on the physical package (formulations change), 
    check the manufacturer's gluten-free info if you're unsure, and look for a certified 
    gluten-free label. This tool cannot detect all cross-contamination risks.
    """
)


def classify_ingredient_list(ingredient_text: str) -> GlutenGuardResult:
    """Classify a raw ingredient list as Green/Yellow/Red for celiac disease safety.

    Args:
        ingredient_text: Raw ingredient list, optionally including allergen/warning
            statements, as plain text.

    Returns:
        A GlutenGuardResult with the classification and a full explanation.
    """
    if not ingredient_text or not ingredient_text.strip():
        raise ValueError("ingredient_text must be non-empty")

    response = client.messages.parse(
        model=model,
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": f"Ingredient list and label text:\n\n{ingredient_text.strip()}",
            }
        ],
        output_format=GlutenGuardResult,
    )

    result = response.parsed_output
    result.mandatory_reminder = MANDATORY_REMINDER
    return result

## Pretty-print the result

In [7]:
CLASSIFICATION_EMOJI = {"Green": "\U0001F7E2", "Yellow": "\U0001F7E1", "Red": "\U0001F534"}


def print_result(result: GlutenGuardResult) -> None:
    icon = CLASSIFICATION_EMOJI.get(result.classification, "\u2753")
    print(f"{icon} {result.classification.upper()}")
    print("-" * 60)
    print(result.summary)

    print("\nINGREDIENT-BY-INGREDIENT")
    print("-" * 60)
    for finding in result.ingredient_analysis:
        print(f"  [{finding.gluten_status}] {finding.ingredient} \u2014 {finding.reasoning}")

    print("\nCROSS-CONTAMINATION")
    print("-" * 60)
    print(result.cross_contamination_warning or "None noted on label")

    print("\nWHY THIS CLASSIFICATION")
    print("-" * 60)
    for reason in result.reasons_for_classification:
        print(f"  - {reason}")

    print("\n\u26A0\uFE0F  " + result.mandatory_reminder)

## Smoke tests

Three canned examples — an obvious Red, an obvious-ish Green, and an ambiguous Yellow — to
sanity-check the engine end to end before wiring up real photos.

In [8]:
test_cases = {
    "obvious_red": """
        Ingredients: Enriched Wheat Flour (Wheat Flour, Niacin, Reduced Iron, Thiamine
        Mononitrate, Riboflavin, Folic Acid), Sugar, Vegetable Oil, Salt.
        Allergen statement: Contains Wheat.
    """,
    "likely_green": """
        Ingredients: Rice, Water, Salt.
        No allergen statement present. Not manufactured in a shared facility.
    """,
    "ambiguous_yellow": """
        Ingredients: Corn Starch, Natural Flavors, Modified Food Starch, Salt, Spices.
        Manufactured in a facility that also processes wheat.
    """,
}

for name, text in test_cases.items():
    print(f"=== {name} ===")
    result = classify_ingredient_list(text)
    print_result(result)
    print()

=== obvious_red ===
🔴 RED
------------------------------------------------------------
This product contains wheat flour as its primary ingredient and carries a 'Contains Wheat' allergen statement, making it unsafe for people with celiac disease.

INGREDIENT-BY-INGREDIENT
------------------------------------------------------------
  [contains_gluten] Enriched Wheat Flour (Wheat Flour, Niacin, Reduced Iron, Thiamine Mononitrate, Riboflavin, Folic Acid) — Wheat flour is an explicit gluten-containing ingredient.
  [gluten_free] Sugar — Plain sugar contains no gluten.
  [gluten_free] Vegetable Oil — Vegetable oil is naturally gluten-free.
  [gluten_free] Salt — Salt is a mineral and contains no gluten.

CROSS-CONTAMINATION
------------------------------------------------------------
Allergen statement: Contains Wheat.

WHY THIS CLASSIFICATION
------------------------------------------------------------
  - The primary ingredient is enriched wheat flour, an explicit gluten-containing grain

## Try your own label text

In [9]:
ingredient_text = input("Paste the ingredient list / label text: ")

result = classify_ingredient_list(ingredient_text)
print_result(result)

🔴 RED
------------------------------------------------------------
This product contains wheat flour in its breading, which is an explicit gluten-containing ingredient, making it unsafe for people with celiac disease.

INGREDIENT-BY-INGREDIENT
------------------------------------------------------------
  [gluten_free] Fresh chicken — Plain chicken meat is naturally gluten-free.
  [contains_gluten] Wheat flour — Wheat flour is a direct gluten-containing grain product.
  [gluten_free] Salt — Salt is a mineral with no gluten.
  [gluten_free] Monosodium glutamate — MSG is typically produced by fermentation and is considered gluten-free.
  [uncertain] Spices — Generic 'spices' can occasionally include anti-caking or blending agents of uncertain source.
  [gluten_free] Dehydrated garlic — Plain dehydrated garlic is naturally gluten-free.
  [gluten_free] Modified milk ingredients — Milk-derived ingredients do not contain gluten.
  [gluten_free] Dried egg white (Egg white, Baker's yeast, Citr